# 내부 데이터 기반 D14 질문 세트 재완료 예측 모델

## 1. 분석 목적과 기준 정의

학교 전체 가입자 40명과 같은 학급 친구 4명 조건을 충족한 사용자를 대상으로,
조건 충족 이후 발생한 첫 질문 세트 완료 시점을 분석 기준점으로 설정한다.

첫 완료 후 24시간 동안 관찰된 행동을 이용하여 정확히 D14에
질문 세트를 다시 완료하는 사용자를 예측한다.

- 분석 단위: 사용자
- 기준 시점: 조건 충족 이후 첫 질문 세트 완료 시각
- 변수 관찰 구간: 첫 완료 직후부터 24시간 이내
- 목표변수: D14 질문 세트 재완료 여부
- 시간 기준: UTC

In [1]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn

print("Python:", sys.version)
print("실행 경로:", sys.executable)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.12.14 (main, Aug 12 2026, 13:57:54) [Clang 21.0.0 (clang-2100.1.1.101)]
실행 경로: /Users/apple/DA15_Part4/sns_service_analysis/.venv312/bin/python
pandas: 3.0.5
scikit-learn: 1.9.1


## 2. 데이터 및 분석 대상 불러오기

In [2]:
from google.cloud import bigquery

PROJECT_ID = "sns-analysis-prj"
DATASET_ID = "sns_analysis"
LOCATION = "asia-northeast3"

client = bigquery.Client(
    project=PROJECT_ID,
    location=LOCATION
)

print("BigQuery 연결 완료")
print("Project:", client.project)

BigQuery 연결 완료
Project: sns-analysis-prj


/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [3]:
from pathlib import Path
import pandas as pd

FINAL_ROOT = Path(
    "/Users/apple/DA15_Part4/git-practice-project/data/raw/final"
)

SCHOOL_IDS = [
    1719, 369, 5372, 4516, 5520,
    4426, 271, 1478, 5491, 352
]

assert FINAL_ROOT.exists()

In [4]:
# 10개 학교 사용자
sql = f"""
SELECT user_id, signup_date, school_id, group_id
FROM `{PROJECT_ID}.{DATASET_ID}.stg_acquisition_user`
WHERE school_id IN UNNEST(@school_ids)
"""

config = bigquery.QueryJobConfig(query_parameters=[
    bigquery.ArrayQueryParameter("school_ids", "INT64", SCHOOL_IDS)
])

users_df = client.query(
    sql, job_config=config, location=LOCATION
).to_dataframe(create_bqstorage_client=False)

print("사용자:", f"{users_df['user_id'].nunique():,}명")

사용자: 5,090명


In [5]:
# 해당 사용자들의 친구 이력
user_ids = users_df["user_id"].dropna().astype(int).unique().tolist()

sql = f"""
SELECT user_id, friend_user_id, friendship_at
FROM `{PROJECT_ID}.{DATASET_ID}.user_friend_count_history`
WHERE user_id IN UNNEST(@user_ids)
"""

config = bigquery.QueryJobConfig(query_parameters=[
    bigquery.ArrayQueryParameter("user_ids", "INT64", user_ids)
])

friends_df = client.query(
    sql, job_config=config, location=LOCATION
).to_dataframe(create_bqstorage_client=False)

print("친구 이력:", f"{len(friends_df):,}건")

친구 이력: 221,675건


In [6]:
# 로컬 덤프의 완료 질문 세트
sets_df = pd.read_parquet(
    FINAL_ROOT / "polls_questionset",
    columns=[
        "id", "user_id", "status",
        "opening_time", "question_piece_id_list"
    ]
)

sets_df = sets_df[
    sets_df["status"].eq("F")
    & sets_df["user_id"].isin(user_ids)
].copy()

print("완료 질문 세트:", f"{len(sets_df):,}건")

완료 질문 세트: 153,411건


In [7]:
# 로컬 덤프의 투표 기록
votes_df = pd.read_parquet(
    FINAL_ROOT / "accounts_userquestionrecord",
    columns=[
        "id", "user_id", "chosen_user_id",
        "question_piece_id", "created_at"
    ]
)

votes_df = votes_df[
    votes_df["user_id"].isin(user_ids)
    | votes_df["chosen_user_id"].isin(user_ids)
].copy()

print("관련 투표 기록:", f"{len(votes_df):,}건")

관련 투표 기록: 1,217,558건


In [8]:
# 시간 형식 통일
assert users_df["user_id"].is_unique

users_df["signup_at"] = pd.to_datetime(
    users_df["signup_date"], utc=True
)
friends_df["friend_at"] = pd.to_datetime(
    friends_df["friendship_at"], utc=True
)

## 3. 학교 40명·동급생 친구 4명 조건 생성

In [9]:
# 학교 40명 도달 시점
school_order = users_df.sort_values(
    ["school_id", "signup_at", "user_id"]
).copy()

school_order["school_order"] = (
    school_order.groupby("school_id").cumcount() + 1
)

school_40_df = (
    school_order.loc[
        school_order["school_order"].eq(40),
        ["school_id", "signup_at"]
    ]
    .rename(columns={"signup_at": "school_40_at"})
)

print("40명 도달 학교:", len(school_40_df), "개")

40명 도달 학교: 10 개


In [10]:
# 같은 학급 친구 연결만 남기기
peers = users_df[["user_id", "group_id"]].rename(
    columns={
        "user_id": "friend_user_id",
        "group_id": "friend_group_id"
    }
)

class_friends_df = (
    friends_df
    .merge(
        users_df[["user_id", "group_id"]],
        on="user_id",
        how="inner"
    )
    .merge(peers, on="friend_user_id", how="inner")
)

class_friends_df = (
    class_friends_df[
        class_friends_df["group_id"].eq(
            class_friends_df["friend_group_id"]
        )
    ]
    .sort_values(["user_id", "friend_at", "friend_user_id"])
    .drop_duplicates(["user_id", "friend_user_id"])
    .copy()
)

print("같은 학급 친구 연결:", f"{len(class_friends_df):,}건")

같은 학급 친구 연결: 55,216건


In [11]:
# 학급 친구 4명 도달 시점
class_friends_df["friend_order"] = (
    class_friends_df.groupby("user_id").cumcount() + 1
)

friend_4_df = (
    class_friends_df.loc[
        class_friends_df["friend_order"].eq(4),
        ["user_id", "friend_at"]
    ]
    .rename(columns={"friend_at": "friend_4_at"})
)

print("학급 친구 4명 도달 사용자:", f"{len(friend_4_df):,}명")

학급 친구 4명 도달 사용자: 4,520명


In [12]:
# 두 기준을 충족한 시점
eligible_df = (
    users_df
    .merge(school_40_df, on="school_id", how="inner")
    .merge(friend_4_df, on="user_id", how="inner")
)

eligible_df["eligible_at"] = eligible_df[
    ["signup_at", "school_40_at", "friend_4_at"]
].max(axis=1)

print("두 기준 충족 사용자:", f"{len(eligible_df):,}명")
display(
    eligible_df[
        ["signup_at", "school_40_at", "friend_4_at", "eligible_at"]
    ].head()
)

두 기준 충족 사용자: 4,520명


,signup_at,school_40_at,friend_4_at,eligible_at
0,2023-05-07 13:49:53.689071+00:00,2023-05-05 04:07:00.053881+00:00,2023-07-13 10:43:27+00:00,2023-07-13 10:43:27+00:00
1,2023-05-12 08:22:57.709702+00:00,2023-05-07 02:57:28.039304+00:00,2023-05-12 09:06:23+00:00,2023-05-12 09:06:23+00:00
2,2023-05-06 06:02:44.599939+00:00,2023-05-05 04:07:00.053881+00:00,2023-05-07 11:25:32+00:00,2023-05-07 11:25:32+00:00
3,2023-05-11 14:20:04.873137+00:00,2023-05-10 23:47:35.094548+00:00,2023-05-12 03:51:06+00:00,2023-05-12 03:51:06+00:00
4,2023-05-13 11:14:18.314353+00:00,2023-05-13 08:00:50.712386+00:00,2023-05-13 11:22:03+00:00,2023-05-13 11:22:03+00:00


## 4. 조건 충족 후 첫 질문 세트 완료 시점 생성

In [13]:
# 시각·질문 조각 ID 형식 통일
sets_df["opening_time"] = pd.to_datetime(
    sets_df["opening_time"], utc=True
)
votes_df["created_at"] = pd.to_datetime(
    votes_df["created_at"], utc=True
)
votes_df["question_piece_id"] = pd.to_numeric(
    votes_df["question_piece_id"], errors="coerce"
).astype("Int64")

In [14]:
# 완료 세트의 질문 조각 펼치기
pieces_df = sets_df[
    ["id", "user_id", "opening_time", "question_piece_id_list"]
].copy()

pieces_df["question_piece_id"] = (
    pieces_df["question_piece_id_list"]
    .astype(str)
    .str.findall(r"\d+")
)

pieces_df = pieces_df.explode("question_piece_id")
pieces_df["question_piece_id"] = pd.to_numeric(
    pieces_df["question_piece_id"], errors="coerce"
).astype("Int64")

pieces_df = pieces_df.drop(columns="question_piece_id_list")
print("질문 조각:", f"{len(pieces_df):,}건")

질문 조각: 1,534,110건


In [15]:
# 세트별 완료 추정 시각
vote_times_df = (
    votes_df
    .groupby(["user_id", "question_piece_id"], as_index=False)
    ["created_at"].max()
)

finish_df = (
    pieces_df
    .merge(
        vote_times_df,
        on=["user_id", "question_piece_id"],
        how="left"
    )
    .groupby(["id", "user_id", "opening_time"], as_index=False)
    ["created_at"].max()
    .rename(columns={"created_at": "finish_at"})
)

finish_df = finish_df[
    finish_df["finish_at"].notna()
    & finish_df["finish_at"].ge(finish_df["opening_time"])
].copy()

print("완료 시각 추정 가능 세트:", f"{len(finish_df):,}건")

완료 시각 추정 가능 세트: 151,566건


In [16]:
# 기준 충족 이후 첫 완료
first_df = eligible_df[
    ["user_id", "eligible_at", "school_id", "group_id"]
].merge(
    finish_df[["id", "user_id", "finish_at"]],
    on="user_id",
    how="inner"
)

first_df = (
    first_df[first_df["finish_at"].ge(first_df["eligible_at"])]
    .sort_values(["user_id", "finish_at", "id"])
    .drop_duplicates("user_id")
    .rename(columns={
        "id": "first_set_id",
        "finish_at": "first_finish_at"
    })
    .copy()
)

print("기준 충족 후 첫 완료 사용자:", f"{len(first_df):,}명")
print(
    "첫 완료 기간:",
    first_df["first_finish_at"].min(),
    "~",
    first_df["first_finish_at"].max()
)

기준 충족 후 첫 완료 사용자: 4,330명
첫 완료 기간: 2023-04-28 12:29:33+00:00 ~ 2023-11-17 08:23:56+00:00


## 5. D14 재완료 라벨 생성

In [17]:
# 예측 시점과 D14 날짜
model_df = first_df.copy()

model_df["predict_at"] = (
    model_df["first_finish_at"] + pd.Timedelta(hours=24)
)

model_df["d14_date"] = (
    model_df["first_finish_at"].dt.normalize()
    + pd.Timedelta(days=14)
)

In [18]:
# 관찰 가능한 사용자만 남기기
observation_end = finish_df["finish_at"].max().floor("D")

model_df = model_df[
    model_df["predict_at"].le(observation_end)
    & (
        model_df["d14_date"] + pd.Timedelta(days=1)
    ).le(observation_end)
].copy()

print("보수적 관찰 종료:", observation_end)
print("모델 대상자:", f"{len(model_df):,}명")

보수적 관찰 종료: 2024-05-07 00:00:00+00:00
모델 대상자: 4,330명


In [19]:
# D14 질문 세트 재완료 라벨
complete_days_df = (
    finish_df[["user_id", "finish_at"]]
    .assign(
        d14_date=lambda df: df["finish_at"].dt.normalize()
    )
    [["user_id", "d14_date"]]
    .drop_duplicates()
)

model_df = model_df.merge(
    complete_days_df.assign(y_d14=1),
    on=["user_id", "d14_date"],
    how="left"
)

model_df["y_d14"] = (
    model_df["y_d14"].fillna(0).astype(int)
)

print("D14 재완료:", f"{model_df['y_d14'].sum():,}명")
print("D14 재완료율:", f"{model_df['y_d14'].mean():.2%}")

D14 재완료: 755명
D14 재완료율: 17.44%


## 6. 첫 완료 후 24시간 행동변수 생성

In [20]:
# 공통 관측 구간
window_df = model_df[
    ["user_id", "first_finish_at", "predict_at"]
].copy()

In [21]:
# 추가 질문 세트 완료
repeat_df = finish_df[
    ["id", "user_id", "finish_at"]
].merge(window_df, on="user_id")

repeat_df = repeat_df[
    repeat_df["finish_at"].gt(repeat_df["first_finish_at"])
    & repeat_df["finish_at"].lt(repeat_df["predict_at"])
]

repeat_features = repeat_df.groupby("user_id").agg(
    repeat_set_count_24h=("id", "nunique"),
    next_finish_at=("finish_at", "min")
).reset_index()

model_df = model_df.merge(
    repeat_features, on="user_id", how="left"
)

model_df["hours_to_next_finish"] = (
    model_df["next_finish_at"] - model_df["first_finish_at"]
).dt.total_seconds() / 3600

In [22]:
# 투표 발신
sent_df = votes_df[
    ["id", "user_id", "chosen_user_id", "created_at"]
].merge(window_df, on="user_id")

sent_df = sent_df[
    sent_df["created_at"].gt(sent_df["first_finish_at"])
    & sent_df["created_at"].lt(sent_df["predict_at"])
]

sent_features = sent_df.groupby("user_id").agg(
    sent_vote_count_24h=("id", "nunique"),
    unique_recipient_count_24h=("chosen_user_id", "nunique")
).reset_index()

model_df = model_df.merge(
    sent_features, on="user_id", how="left"
)

In [23]:
# 투표 수신
received_df = (
    votes_df[
        ["id", "user_id", "chosen_user_id", "created_at"]
    ]
    .rename(columns={
        "user_id": "sender_id",
        "chosen_user_id": "user_id"
    })
    .merge(window_df, on="user_id")
)

received_df = received_df[
    received_df["created_at"].gt(
        received_df["first_finish_at"]
    )
    & received_df["created_at"].lt(
        received_df["predict_at"]
    )
]

received_features = received_df.groupby("user_id").agg(
    received_vote_count_24h=("id", "nunique"),
    unique_sender_count_24h=("sender_id", "nunique")
).reset_index()

model_df = model_df.merge(
    received_features, on="user_id", how="left"
)

In [24]:
# 같은 학급 친구 관계
friend_window_df = class_friends_df[
    ["user_id", "friend_at"]
].merge(window_df, on="user_id")

friend_window_df["friends_at_first"] = (
    friend_window_df["friend_at"]
    .le(friend_window_df["first_finish_at"])
)

friend_window_df["new_friends_24h"] = (
    friend_window_df["friend_at"]
    .gt(friend_window_df["first_finish_at"])
    & friend_window_df["friend_at"]
    .lt(friend_window_df["predict_at"])
)

friend_features = friend_window_df.groupby("user_id").agg(
    class_friends_at_first=("friends_at_first", "sum"),
    new_class_friends_24h=("new_friends_24h", "sum")
).reset_index()

model_df = model_df.merge(
    friend_features, on="user_id", how="left"
)

In [25]:
# 결측 처리와 확인
count_cols = [
    "repeat_set_count_24h",
    "sent_vote_count_24h",
    "unique_recipient_count_24h",
    "received_vote_count_24h",
    "unique_sender_count_24h",
    "class_friends_at_first",
    "new_class_friends_24h",
]

model_df[count_cols] = (
    model_df[count_cols].fillna(0).astype(int)
)

print("사용자:", f"{len(model_df):,}명")
print("사용자 ID 중복:", model_df["user_id"].duplicated().sum())
print("첫 완료 당시 학급 친구 4명 미만:",
      model_df["class_friends_at_first"].lt(4).sum())

display(
    model_df[count_cols + ["hours_to_next_finish", "y_d14"]]
    .describe()
)

사용자: 4,330명
사용자 ID 중복: 0
첫 완료 당시 학급 친구 4명 미만: 0


,repeat_set_count_24h,sent_vote_count_24h,unique_recipient_count_24h,received_vote_count_24h,unique_sender_count_24h,class_friends_at_first,new_class_friends_24h,hours_to_next_finish,y_d14
count,4330.000000,4330.000000,4330.000000,4330.000000,4330.000000,4330.000000,4330.000000,3967.000000,4330.000000
mean,5.908083,47.345958,18.525173,38.018476,14.787991,5.678753,3.234873,4.337036,0.174365
std,4.407081,38.534513,11.993586,32.644556,10.347300,2.250235,3.001429,5.175151,0.379467
min,0.000000,0.000000,0.000000,0.000000,0.000000,4.000000,0.000000,0.023056,0.000000
25%,2.000000,17.000000,10.000000,15.000000,7.000000,4.000000,1.000000,1.005278,0.000000
50%,5.000000,39.000000,17.000000,30.000000,13.000000,5.000000,3.000000,1.842778,0.000000
75%,9.000000,69.000000,26.000000,52.000000,20.000000,6.000000,5.000000,6.027361,0.000000
max,24.000000,221.000000,76.000000,316.000000,89.000000,23.000000,22.000000,23.972222,1.000000


In [26]:
# 활성화까지 걸린 시간
model_df = model_df.merge(
    eligible_df[["user_id", "signup_at"]],
    on="user_id",
    how="left",
    validate="one_to_one"
)

model_df["signup_to_eligible_hours"] = (
    model_df["eligible_at"] - model_df["signup_at"]
).dt.total_seconds() / 3600

model_df["eligible_to_first_hours"] = (
    model_df["first_finish_at"] - model_df["eligible_at"]
).dt.total_seconds() / 3600

In [27]:
# 첫 완료 시점
model_df["first_month"] = (
    model_df["first_finish_at"].dt.strftime("%Y-%m")
)
model_df["first_weekday"] = (
    model_df["first_finish_at"].dt.dayofweek
)
model_df["first_hour"] = (
    model_df["first_finish_at"].dt.hour
)

## 7. 분석 대상과 D14 재완료 분포 확인

In [28]:
# 월별 모델 표본과 D14 결과
monthly_df = model_df.groupby("first_month").agg(
    users=("user_id", "nunique"),
    d14_users=("y_d14", "sum"),
    d14_rate=("y_d14", "mean")
)

monthly_df["d14_rate"] = (
    monthly_df["d14_rate"] * 100
).round(2)

display(monthly_df)

,users,d14_users,d14_rate
first_month,,,
2023-04,285,56,19.65
2023-05,4016,697,17.36
2023-06,24,2,8.33
2023-07,1,0,0.00
2023-08,1,0,0.00
2023-09,2,0,0.00
2023-11,1,0,0.00


In [29]:
# 5월 첫 완료일별 표본
may_daily_df = (
    model_df[model_df["first_month"].eq("2023-05")]
    .assign(
        first_date=lambda df: (
            df["first_finish_at"].dt.strftime("%Y-%m-%d")
        )
    )
    .groupby("first_date")
    .agg(
        users=("user_id", "size"),
        d14_users=("y_d14", "sum")
    )
)

may_daily_df["d14_rate"] = (
    may_daily_df["d14_users"]
    / may_daily_df["users"] * 100
).round(2)

display(may_daily_df)

,users,d14_users,d14_rate
first_date,,,
2023-05-01,42,7,16.67
2023-05-02,20,1,5.0
2023-05-03,251,50,19.92
2023-05-04,163,16,9.82
2023-05-05,380,46,12.11
2023-05-06,203,25,12.32
2023-05-07,106,16,15.09
2023-05-08,110,20,18.18
2023-05-09,125,36,28.8


## 8. 학습·검증·테스트 데이터 분리

In [30]:
# 시간순 분리
t = model_df["first_finish_at"]

train_df = model_df[
    t.lt(pd.Timestamp("2023-05-13", tz="UTC"))
].copy()

valid_df = model_df[
    t.ge(pd.Timestamp("2023-05-13", tz="UTC"))
    & t.lt(pd.Timestamp("2023-05-18", tz="UTC"))
].copy()

test_df = model_df[
    t.ge(pd.Timestamp("2023-05-18", tz="UTC"))
    & t.lt(pd.Timestamp("2023-06-01", tz="UTC"))
].copy()

In [31]:
# 분리 결과 확인
split_df = pd.DataFrame({
    "학습": [len(train_df), train_df["y_d14"].sum()],
    "검증": [len(valid_df), valid_df["y_d14"].sum()],
    "테스트": [len(test_df), test_df["y_d14"].sum()],
}, index=["사용자", "D14 재완료"])

display(split_df)

for name, df in [
    ("학습", train_df),
    ("검증", valid_df),
    ("테스트", test_df),
]:
    print(name, "학교 수:", df["school_id"].nunique())

,학습,검증,테스트
사용자,2431,1016,854
D14 재완료,405,197,151


학습 학교 수: 8
검증 학교 수: 9
테스트 학교 수: 10


In [32]:
# 학교별 분할 인원
school_split_df = pd.concat([
    train_df.assign(split="학습"),
    valid_df.assign(split="검증"),
    test_df.assign(split="테스트"),
])

display(
    pd.crosstab(
        school_split_df["school_id"],
        school_split_df["split"]
    )
)

split,검증,테스트,학습
school_id,,,
271,3,3,398
352,8,13,360
369,7,4,452
1478,109,88,203
1719,373,147,0
4426,0,425,0
4516,17,8,387
5372,209,48,187
5491,189,21,211


In [33]:
# 학교별 시간순 분리
ordered_df = model_df.sort_values(
    ["school_id", "first_finish_at", "user_id"]
).copy()

school_order = (
    ordered_df.groupby("school_id").cumcount() + 1
)
school_size = (
    ordered_df.groupby("school_id")["user_id"]
    .transform("size")
)

ordered_df["split"] = pd.cut(
    school_order / school_size,
    bins=[0, 0.6, 0.8, 1],
    labels=["학습", "검증", "테스트"],
    include_lowest=True
)

train_df = ordered_df[ordered_df["split"].eq("학습")].copy()
valid_df = ordered_df[ordered_df["split"].eq("검증")].copy()
test_df = ordered_df[ordered_df["split"].eq("테스트")].copy()

In [34]:
# 새 분할 확인
display(
    ordered_df.groupby("split", observed=True).agg(
        users=("user_id", "size"),
        d14_users=("y_d14", "sum"),
        d14_rate=("y_d14", "mean")
    )
)

display(
    pd.crosstab(
        ordered_df["school_id"],
        ordered_df["split"]
    )
)

,users,d14_users,d14_rate
split,,,
학습,2595,483,0.186127
검증,865,161,0.186127
테스트,870,111,0.127586


split,학습,검증,테스트
school_id,,,
271,243,81,82
352,231,77,77
369,278,93,93
1478,244,81,82
1719,312,104,104
4426,256,86,86
4516,250,83,84
5372,268,89,90
5491,253,84,85


## 9. 예측모델 학습 및 검증 성능 비교

In [35]:
# 모델 입력변수
import numpy as np

NUMERIC = [
    "repeat_set_count_24h",
    "hours_to_next_finish",
    "sent_vote_count_24h",
    "unique_recipient_count_24h",
    "received_vote_count_24h",
    "unique_sender_count_24h",
    "class_friends_at_first",
    "new_class_friends_24h",
    "signup_to_eligible_hours",
    "eligible_to_first_hours",
    "first_weekday",
    "first_hour",
]

FEATURES = NUMERIC + ["school_id"]

for df in [train_df, valid_df, test_df]:
    df["hours_to_next_finish"] = (
        df["hours_to_next_finish"].fillna(24)
    )
    assert df[FEATURES].notna().all().all()

In [36]:
# 공통 전처리
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import clone

preprocess = ColumnTransformer([
    ("numeric", StandardScaler(), NUMERIC),
    ("school", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ), ["school_id"]),
])

In [37]:
# 비교할 모델
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
)

models = {
    "로지스틱 무규제": LogisticRegression(
        C=np.inf, max_iter=3000
    ),
    "로지스틱 L1": LogisticRegression(
        C=1, l1_ratio=1, solver="liblinear", max_iter=3000
    ),
    "로지스틱 L2": LogisticRegression(
        C=1, l1_ratio=0, max_iter=3000
    ),
    "랜덤포레스트": RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=15,
        random_state=42,
        n_jobs=-1,
    ),
    "그래디언트 부스팅": HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=7,
        min_samples_leaf=25,
        random_state=42,
    ),
}

try:
    from xgboost import XGBClassifier

    models["XGBoost"] = XGBClassifier(
        n_estimators=250,
        max_depth=3,
        learning_rate=0.05,
        min_child_weight=10,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=4,
    )
except ImportError:
    print("XGBoost가 설치되지 않아 나머지 모델부터 비교합니다.")

In [38]:
# 학습과 검증 성능 비교
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
)

X_train, y_train = train_df[FEATURES], train_df["y_d14"]
X_valid, y_valid = valid_df[FEATURES], valid_df["y_d14"]

fitted_models = {}
results = [{
    "model": "상수 기준",
    "PR-AUC": y_valid.mean(),
    "ROC-AUC": 0.5,
    "Brier": brier_score_loss(
        y_valid, np.repeat(y_train.mean(), len(y_valid))
    ),
}]

for name, estimator in models.items():
    pipe = Pipeline([
        ("preprocess", clone(preprocess)),
        ("model", estimator),
    ])
    pipe.fit(X_train, y_train)
    prob = pipe.predict_proba(X_valid)[:, 1]

    fitted_models[name] = pipe
    results.append({
        "model": name,
        "PR-AUC": average_precision_score(y_valid, prob),
        "ROC-AUC": roc_auc_score(y_valid, prob),
        "Brier": brier_score_loss(y_valid, prob),
    })

validation_results = pd.DataFrame(results)
display(validation_results.sort_values("PR-AUC", ascending=False))

,model,PR-AUC,ROC-AUC,Brier
6,XGBoost,0.321190,0.639990,0.145583
4,랜덤포레스트,0.306325,0.647224,0.144743
5,그래디언트 부스팅,0.302432,0.640960,0.145814
1,로지스틱 무규제,0.296000,0.628406,0.147085
3,로지스틱 L2,0.295919,0.628167,0.147008
2,로지스틱 L1,0.295677,0.627567,0.146993
0,상수 기준,0.186127,0.500000,0.151484


| 지표 | 뜻 | 이 결과의 해석 |
| --- | --- | --- |
| **PR-AUC** | 재완료자를 높은 점수에 얼마나 잘 모으는지. **높을수록 좋음** | XGBoost **0.321**로 1위, 랜덤포레스트 **0.306**으로 2위. 검증 데이터의 재완료율인 **0.186**이 상수 기준선. `0.321`은 “정확도 32.1%”라는 뜻은 아님. |
| **ROC-AUC** | 재완료자에게 미재완료자보다 높은 점수를 줄 확률. **높을수록 좋음** | 랜덤포레스트 **0.647**, XGBoost **0.640**. 구분 능력은 있지만 강한 수준은 아님. 무작위 예측은 0.5 |
| **Brier Score** | 예측 확률과 실제 결과의 평균 제곱 오차. **낮을수록 좋음** | 랜덤포레스트 **0.1447**, XGBoost **0.1456**, 상수 기준 **0.1515**. 확률 예측의 개선 폭은 크지 않음. 이것만으로 확률 보정이 잘됐다고 단정할 수는 없음. |

In [39]:
# 재완료 예측 확률이 가장 낮은 10%·20% 확인
rows = []
y = y_valid.to_numpy()

for name in ["XGBoost", "랜덤포레스트"]:
    prob = fitted_models[name].predict_proba(X_valid)[:, 1]
    order = np.argsort(prob)

    for share in [0.1, 0.2]:
        selected = order[:int(np.ceil(len(y) * share))]
        non_return = (y[selected] == 0).sum()

        rows.append({
            "모델": name,
            "위험군": f"상위 {share:.0%}",
            "선정 인원": len(selected),
            "실제 미재완료율": non_return / len(selected),
            "미재완료자 포착률": non_return / (y == 0).sum(),
            "기준 대비 Lift": (
                non_return / len(selected)
            ) / (y == 0).mean(),
        })

display(pd.DataFrame(rows))
print("전체 미재완료율:", f"{(y == 0).mean():.2%}")

,모델,위험군,선정 인원,실제 미재완료율,미재완료자 포착률,기준 대비 Lift
0,XGBoost,상위 10%,87,0.954023,0.117898,1.172202
1,XGBoost,상위 20%,173,0.913295,0.224432,1.122159
2,랜덤포레스트,상위 10%,87,0.931034,0.115057,1.143956
3,랜덤포레스트,상위 20%,173,0.890173,0.218750,1.093750


전체 미재완료율: 81.39%


| 위험군 | XGBoost 실제 미재완료율 | 랜덤포레스트 |
| --- | ---: | ---: |
| 상위 10% | **95.4%** (87명 중 83명) | 93.1% (87명 중 81명) |
| 상위 20% | **91.3%** (173명 중 158명) | 89.0% (173명 중 154명) |

- 전체 미재완료율이 **81.39%**라서 XGBoost의 상위 10%는 전체보다 14.0%포인트 높다
- 다만 두 모델의 차이는 2~4명으로 작아서 XGBoost를 최종 후보로 선택하고 테스트 데이터에서 한 번 확인할 것.

## 10. 최종 모델 테스트 성능 평가

In [40]:
# 검증에서 선택한 XGBoost를 학습+검증 데이터로 다시 학습
train_valid = pd.concat([train_df, valid_df])

final_model = clone(fitted_models["XGBoost"])
final_model.fit(
    train_valid[FEATURES],
    train_valid["y_d14"]
)

y_test = test_df["y_d14"].to_numpy()
prob_test = final_model.predict_proba(
    test_df[FEATURES]
)[:, 1]

In [41]:
print("테스트 재완료율:", f"{y_test.mean():.2%}")
print("PR-AUC:", round(average_precision_score(y_test, prob_test), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, prob_test), 4))
print("Brier:", round(brier_score_loss(y_test, prob_test), 4))

rows = []
for share in [0.1, 0.2]:
    selected = np.argsort(prob_test)[:int(np.ceil(len(y_test) * share))]
    non_return = (y_test[selected] == 0).sum()

    rows.append({
        "위험군": f"상위 {share:.0%}",
        "선정 인원": len(selected),
        "실제 미재완료율": non_return / len(selected),
        "미재완료자 포착률": non_return / (y_test == 0).sum(),
        "기준 대비 Lift": (
            non_return / len(selected)
        ) / (y_test == 0).mean(),
    })

display(pd.DataFrame(rows))
print("전체 미재완료율:", f"{(y_test == 0).mean():.2%}")

테스트 재완료율: 12.76%
PR-AUC: 0.2269
ROC-AUC: 0.6654
Brier: 0.1129


,위험군,선정 인원,실제 미재완료율,미재완료자 포착률,기준 대비 Lift
0,상위 10%,87,0.965517,0.110672,1.106719
1,상위 20%,174,0.942529,0.216074,1.080369


전체 미재완료율: 87.24%


| 지표 | 테스트 결과 | 해석 |
| --- | ---: | --- |
| PR-AUC | **0.2269** | 테스트 재완료율 **0.1276**보다 높음. 검증의 0.321과는 양성 비율이 달라 숫자만 직접 비교하긴 어렵다. |
| ROC-AUC | **0.6654** | 어느 정도 구분하지만 강한 예측력은 아님. |
| 위험 상위 10% | **87명 중 84명 미재완료** | 96.55%로 높지만, 전체도 87.24%가 미재완료. 미재완료자 전체의 **11.07%**를 포착. |
| 위험 상위 20% | **174명 중 164명 미재완료** | 미재완료자 전체의 **21.61%**를 포착. |


- 즉, 상위 10%를 골랐을 때 실제 미재완료율이 전체보다 9.31%포인트 높다. 
- 발표에서는 “위험군을 어느 정도 구분했다”까지 말할 수 있고, 개입 대상을 크게 줄일 만큼 강하게 선별한다고 말하기는 어렵다.
- 다음은 예측 확률 자체가 맞는지 확인. Brier 0.1129는 낮아 보일 수 있지만 상수 예측과 비교.

In [43]:
# 상수 예측과 확률 보정 확인
base_prob = train_valid["y_d14"].mean()
print("상수 기준 Brier:",
      round(brier_score_loss(
          y_test, np.full(len(y_test), base_prob)
      ), 4))

calibration_df = pd.DataFrame({
    "예측확률": prob_test,
    "실제결과": y_test,
})
calibration_df["확률구간"] = pd.qcut(
    calibration_df["예측확률"], 5, duplicates="drop"
)

display(
    calibration_df.groupby("확률구간", observed=True).agg(
        인원=("실제결과", "size"),
        평균_예측확률=("예측확률", "mean"),
        실제_재완료율=("실제결과", "mean"),
    )
)

상수 기준 Brier: 0.1147


,인원,평균_예측확률,실제_재완료율
확률구간,,,
"(0.019, 0.0931]",174,0.067091,0.057471
"(0.0931, 0.134]",174,0.112626,0.074713
"(0.134, 0.182]",174,0.157094,0.097701
"(0.182, 0.269]",174,0.219627,0.178161
"(0.269, 0.692]",174,0.379019,0.229885


## 11. 변수군 기여도 및 기준 모델 비교

### 11-1. 변수군 제외 분석

In [44]:
# 변수군과 평가 함수
feature_groups = {
    "반복 완료": [
        "repeat_set_count_24h", "hours_to_next_finish"
    ],
    "보낸 투표": [
        "sent_vote_count_24h", "unique_recipient_count_24h"
    ],
    "받은 투표": [
        "received_vote_count_24h", "unique_sender_count_24h"
    ],
    "학급 친구": [
        "class_friends_at_first", "new_class_friends_24h"
    ],
    "활성화·시점": [
        "signup_to_eligible_hours", "eligible_to_first_hours",
        "first_weekday", "first_hour", "school_id"
    ],
}

def evaluate_valid(prob):
    n = int(np.ceil(len(prob) * 0.2))
    selected = np.argsort(prob)[:n]

    return {
        "PR-AUC": average_precision_score(y_valid, prob),
        "위험 상위 20% 미재완료율": (
            y_valid.iloc[selected].eq(0).mean()
        ),
    }

In [45]:
# 하나씩 제외하고 비교
full_prob = fitted_models["XGBoost"].predict_proba(X_valid)[:, 1]
rows = [{"제외 변수군": "없음", **evaluate_valid(full_prob)}]

for group_name, excluded in feature_groups.items():
    kept = [col for col in FEATURES if col not in excluded]
    numeric = [col for col in kept if col in NUMERIC]

    transformers = [("numeric", StandardScaler(), numeric)]
    if "school_id" in kept:
        transformers.append((
            "school",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ["school_id"],
        ))

    pipe = Pipeline([
        ("preprocess", ColumnTransformer(transformers)),
        ("model", clone(models["XGBoost"])),
    ])
    pipe.fit(X_train[kept], y_train)

    prob = pipe.predict_proba(X_valid[kept])[:, 1]
    rows.append({
        "제외 변수군": group_name,
        **evaluate_valid(prob),
    })

ablation_df = pd.DataFrame(rows)
ablation_df["PR-AUC 감소"] = (
    ablation_df.loc[0, "PR-AUC"] - ablation_df["PR-AUC"]
)

display(ablation_df.round(4))

,제외 변수군,PR-AUC,위험 상위 20% 미재완료율,PR-AUC 감소
0,없음,0.3212,0.9133,0.0000
1,반복 완료,0.2910,0.9191,0.0301
2,보낸 투표,0.2940,0.9133,0.0272
3,받은 투표,0.3127,0.8902,0.0085
4,학급 친구,0.3181,0.9017,0.0031
5,활성화·시점,0.2341,0.8324,0.0871


| 뺀 변수군 | PR-AUC 변화 | 위험 상위 20%에서 보이는 점 |
| --- | --- | --- |
| 반복 완료 | **0.0301 감소** | 미재완료율은 오히려 91.33% → 91.91% |
| 보낸 투표 | **0.0272 감소** | 미재완료율은 동일 |
| 받은 투표 | 0.0085 감소 | 미재완료율은 **89.02%**로 하락 |
| 학급 친구 | 0.0031 감소 | 미재완료율은 **90.17%**로 하락 |

- 즉, 반복 완료와 보낸 투표는 재완료자 구분에 정보를 더했고, 받은 투표와 친구 정보는 이번 검증 표본에서 추가 정보가 작았다. 
- 다만 위험 상위 20%만 보면 반복 완료를 뺐을 때 오히려 1명이 더 정확히 선정된다. 한 지표만으로 “반복 완료가 위험군 선정에 가장 중요하다”고 말하면 안 되는 이유.
- 특히 ‘활성화·시점’에는 학교, 가입부터 기준 충족까지의 시간, 기준 충족부터 첫 완료까지의 시간, 첫 완료 요일·시간이 한꺼번에 들어 있음. 
- 지금 표만으로는 활성화 속도가 중요한지, 학교 차이인지, 시간대 차이인지 알 수 없기 때문에, 이 묶음을 나눠서 확인.

In [47]:
# 기존 결과 보관
ablation_main_df = ablation_df.copy()

feature_groups = {
    "학교": ["school_id"],
    "가입→기준 충족 시간": ["signup_to_eligible_hours"],
    "기준 충족→첫 완료 시간": ["eligible_to_first_hours"],
    "첫 완료 요일·시간": ["first_weekday", "first_hour"],
}

In [48]:
# 하나씩 제외하고 비교
full_prob = fitted_models["XGBoost"].predict_proba(X_valid)[:, 1]
rows = [{"제외 변수군": "없음", **evaluate_valid(full_prob)}]

for group_name, excluded in feature_groups.items():
    kept = [col for col in FEATURES if col not in excluded]
    numeric = [col for col in kept if col in NUMERIC]

    transformers = [("numeric", StandardScaler(), numeric)]
    if "school_id" in kept:
        transformers.append((
            "school",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ["school_id"],
        ))

    pipe = Pipeline([
        ("preprocess", ColumnTransformer(transformers)),
        ("model", clone(models["XGBoost"])),
    ])
    pipe.fit(X_train[kept], y_train)

    prob = pipe.predict_proba(X_valid[kept])[:, 1]
    rows.append({
        "제외 변수군": group_name,
        **evaluate_valid(prob),
    })

ablation_df = pd.DataFrame(rows)
ablation_df["PR-AUC 감소"] = (
    ablation_df.loc[0, "PR-AUC"] - ablation_df["PR-AUC"]
)

display(ablation_df.round(4))

,제외 변수군,PR-AUC,위험 상위 20% 미재완료율,PR-AUC 감소
0,없음,0.3212,0.9133,0.0000
1,학교,0.2759,0.8671,0.0453
2,가입→기준 충족 시간,0.3027,0.9075,0.0185
3,기준 충족→첫 완료 시간,0.3136,0.9017,0.0076
4,첫 완료 요일·시간,0.2895,0.9075,0.0317


| 뺀 변수 | PR-AUC 감소 | 위험 상위 20% 미재완료율 |
| --- | ---: | ---: |
| 학교 | **0.0453** | **91.33% → 86.71%** |
| 첫 완료 요일·시간 | 0.0317 | 90.75% |
| 가입→기준 충족 시간 | 0.0185 | 90.75% |
| 기준 충족→첫 완료 시간 | 0.0076 | 90.17% |

- 따라서 앞에서 ‘활성화·시점’ 묶음이 크게 중요했던 결과를 **“활성화가 빠를수록 유지된다”**로 해석하면 안 된다.
- 그 묶음 안에서는 학교와 첫 완료 요일·시간의 예측 기여가 더 컸음. 가입부터 기준 충족까지의 시간도 정보가 있지만, 기준 충족 후 첫 완료까지의 시간은 추가 기여가 작았음.
- 이건 학교마다 D14 재완료 수준이나 이용 환경이 다를 가능성을 보여주는 예측 결과. 
- 학교가 원인이라는 뜻은 아니고, 현재 모델은 이미 학습에 포함된 10개 학교의 후속 사용자를 평가한 결과라는 제한도 있음.
- 다음엔 학교별 재완료율이 실제로 어떻게 다른지 볼 예정.

In [49]:
display(
    ordered_df.groupby(
        ["school_id", "split"], observed=True
    ).agg(
        사용자=("user_id", "size"),
        D14_재완료율=("y_d14", "mean"),
    ).round(3)
)

사용자  D14_재완료율
school_id split               
271       학습     243     0.198
          검증      81     0.185
          테스트     82     0.073
352       학습     231     0.056
          검증      77     0.117
          테스트     77     0.104
369       학습     278     0.137
          검증      93     0.086
          테스트     93     0.075
1478      학습     244     0.197
          검증      81     0.333
          테스트     82     0.159
1719      학습     312     0.176
          검증     104     0.067
          테스트    104     0.077
4426      학습     256     0.270
          검증      86     0.174
          테스트     86     0.128
4516      학습     250     0.224
          검증      83     0.241
          테스트     84     0.190
5372      학습     268     0.131
          검증      89     0.191
          테스트     90     0.178
5491      학습     253     0.134
          검증      84     0.143
          테스트     85     0.129
5520      학습     260     0.335
          검증      87     0.356
          테스트     87     0.172

학교마다 차이도 있고, 같은 학교 안에서도 후반 사용자로 갈수록 재완료율이 달라짐
- 학교 5520은 학습 33.5% → 검증 35.6% → 테스트 **17.2%**로 떨어짐.
- 271도 19.8% → 18.5% → 7.3%, 4426은 27.0% → 17.4% → **12.8%**.
- 검증에서 테스트로 넘어갈 때 10개 학교 중 9개의 재완료율이 내려감.
- 그래서 테스트 전체 재완료율이 **12.76%**로 낮아지고, 모델이 확률을 높게 예측한 현상과도 맞아떨어짐. 다만 이 표만으로 왜 내려갔는지는 알 수 없음.
- 학교 ID는 예측에 도움이 되지만, 학교별 비율이 계속 고정돼 있다고 가정하면 안 되겠다는 결과.
다음으로 **“학교와 첫 완료 시점만 알아도 비슷하게 맞히는가, 아니면 첫 24시간 행동이 실제로 더해주는 정보가 있는가?”**

### 11-2. 학교·시점 기준 모델과 전체 변수 모델 비교

In [50]:
# 학교·첫 완료 요일/시간만 사용하는 기준 모델
base_features = ["school_id", "first_weekday", "first_hour"]

base_preprocess = ColumnTransformer([
    ("numeric", StandardScaler(),
     ["first_weekday", "first_hour"]),
    ("school", OneHotEncoder(
        handle_unknown="ignore", sparse_output=False
    ), ["school_id"]),
])

base_model = Pipeline([
    ("preprocess", base_preprocess),
    ("model", clone(models["XGBoost"])),
])

base_model.fit(
    train_df[base_features], y_train
)

base_prob = base_model.predict_proba(
    valid_df[base_features]
)[:, 1]

display(pd.DataFrame([
    {"모델": "학교·시점만", **evaluate_valid(base_prob)},
    {"모델": "전체 변수", **evaluate_valid(full_prob)},
]).round(4))

,모델,PR-AUC,위험 상위 20% 미재완료율
0,학교·시점만,0.2581,0.9133
1,전체 변수,0.3212,0.9133


| 검증 결과 | 학교·시점만 | 전체 변수 |
| --- | ---: | ---: |
| PR-AUC | 0.2581 | **0.3212** |
| 위험 상위 20% 실제 미재완료율 | **91.33%** | **91.33%** |

- PR-AUC는 0.0631 올랐음. 즉 첫 24시간 행동에 추가 예측 정보가 있음. 
- 그런데 173명을 개입 대상으로 고르는 상황에서는 두 모델 모두 같은 수의 미재완료자를 고름. 그래서 “초기 행동을 추가하면 개입 대상 선정이 더 좋아진다”는 결론은 아직 못 내림.
- 다음으로 두 모델이 같은 사람을 골랐는지, 그리고 상위 10%에서는 차이가 나는지 확인

In [51]:
y = y_valid.to_numpy()
rows = []

for share in [0.1, 0.2]:
    n = int(np.ceil(len(y) * share))
    base_idx = np.argsort(base_prob)[:n]
    full_idx = np.argsort(full_prob)[:n]

    rows.append({
        "위험군": f"상위 {share:.0%}",
        "선정 인원": n,
        "학교·시점 미재완료자": (y[base_idx] == 0).sum(),
        "전체 변수 미재완료자": (y[full_idx] == 0).sum(),
        "두 모델 공통 선정": len(
            set(base_idx) & set(full_idx)
        ),
    })

display(pd.DataFrame(rows))

,위험군,선정 인원,학교·시점 미재완료자,전체 변수 미재완료자,두 모델 공통 선정
0,상위 10%,87,79,83,36
1,상위 20%,173,158,158,87


첫 완료 후 24시간 행동은 학교·시점만 쓴 모델보다 재완료자를 구분하는 데 도움이 됐다(PR-AUC 0.258 → 0.321). 검증 데이터에서 개입 대상을 상위 10%로 좁히면 미재완료자 4명을 더 찾았지만, 상위 20%에서는 추가 포착이 없었다.

## 12. 테스트셋 기준 모델과 전체 모델 비교

학교와 이용 시점 등 사용자의 기본 조건만 사용한 기준 모델과,
첫 질문 세트 완료 후 24시간 행동을 추가한 전체 모델을 비교한다.

이를 통해 첫 24시간 행동이 D14 재완료 예측에 추가 정보를 제공하는지
최종 테스트 데이터에서 확인한다.

In [52]:
# 비교용 데이터와 변수 정의
# 기존 데이터를 변경하지 않도록 복사
train_compare = train_df.copy()
valid_compare = valid_df.copy()
test_compare = test_df.copy()

# 최초 완료 날짜를 연속된 숫자로 변환
# 학습 데이터의 최초 날짜를 0일로 설정
time_origin = train_compare["first_finish_at"].min().floor("D")

for df in [train_compare, valid_compare, test_compare]:
    df["first_day_index"] = (
        df["first_finish_at"] - time_origin
    ).dt.total_seconds() / 86400

    # 24시간 안에 추가 완료가 없는 사용자는 24시간으로 처리
    df["hours_to_next_finish"] = (
        df["hours_to_next_finish"].fillna(24)
    )

# 첫 완료 시점까지 이미 알고 있는 기본 정보
CONTEXT_NUMERIC = [
    "first_day_index",
    "signup_to_eligible_hours",
    "eligible_to_first_hours",
    "first_weekday",
    "first_hour",
    "class_friends_at_first",
]

# 첫 완료 후 24시간 동안 관찰한 행동
BEHAVIOR_NUMERIC = [
    "repeat_set_count_24h",
    "hours_to_next_finish",
    "sent_vote_count_24h",
    "unique_recipient_count_24h",
    "received_vote_count_24h",
    "unique_sender_count_24h",
    "new_class_friends_24h",
]

CONTEXT_FEATURES = CONTEXT_NUMERIC + ["school_id"]

FULL_FEATURES = (
    CONTEXT_NUMERIC
    + BEHAVIOR_NUMERIC
    + ["school_id"]
)

print("기준 모델 변수:", CONTEXT_FEATURES)
print("추가 행동변수:", BEHAVIOR_NUMERIC)

기준 모델 변수: ['first_day_index', 'signup_to_eligible_hours', 'eligible_to_first_hours', 'first_weekday', 'first_hour', 'class_friends_at_first', 'school_id']
추가 행동변수: ['repeat_set_count_24h', 'hours_to_next_finish', 'sent_vote_count_24h', 'unique_recipient_count_24h', 'received_vote_count_24h', 'unique_sender_count_24h', 'new_class_friends_24h']


In [53]:
# 두 모델 학습
def make_xgb_pipeline(numeric_cols):
    preprocess_compare = ColumnTransformer([
        (
            "numeric",
            StandardScaler(),
            numeric_cols,
        ),
        (
            "school",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            ["school_id"],
        ),
    ])

    return Pipeline([
        ("preprocess", preprocess_compare),
        ("model", clone(models["XGBoost"])),
    ])


# 최종 테스트 전 학습+검증 데이터 결합
train_valid_compare = pd.concat(
    [train_compare, valid_compare],
    ignore_index=True,
)

# 기준 모델
context_model = make_xgb_pipeline(CONTEXT_NUMERIC)
context_model.fit(
    train_valid_compare[CONTEXT_FEATURES],
    train_valid_compare["y_d14"],
)

# 24시간 행동 포함 모델
behavior_model = make_xgb_pipeline(
    CONTEXT_NUMERIC + BEHAVIOR_NUMERIC
)
behavior_model.fit(
    train_valid_compare[FULL_FEATURES],
    train_valid_compare["y_d14"],
)

# 동일한 테스트셋 예측
y_test_compare = test_compare["y_d14"].to_numpy()

context_prob_test = context_model.predict_proba(
    test_compare[CONTEXT_FEATURES]
)[:, 1]

behavior_prob_test = behavior_model.predict_proba(
    test_compare[FULL_FEATURES]
)[:, 1]

print("두 모델 학습 및 테스트 예측 완료")

두 모델 학습 및 테스트 예측 완료


In [54]:
# 테스트 성능 비교
comparison_rows = []

for model_name, prob in [
    ("기준 모델", context_prob_test),
    ("24시간 행동 포함 모델", behavior_prob_test),
]:
    comparison_rows.append({
        "모델": model_name,
        "PR-AUC": average_precision_score(
            y_test_compare, prob
        ),
        "ROC-AUC": roc_auc_score(
            y_test_compare, prob
        ),
        "Brier Score": brier_score_loss(
            y_test_compare, prob
        ),
    })

test_model_comparison = pd.DataFrame(comparison_rows)

display(
    test_model_comparison.round(4)
)

print(
    "테스트 D14 재완료율:",
    f"{y_test_compare.mean():.2%}"
)

,모델,PR-AUC,ROC-AUC,Brier Score
0,기준 모델,0.1574,0.5735,0.1203
1,24시간 행동 포함 모델,0.2201,0.6489,0.1133


테스트 D14 재완료율: 12.76%


In [55]:
# 위험 상위 10%·20% 비교
risk_rows = []

for model_name, prob in [
    ("기준 모델", context_prob_test),
    ("24시간 행동 포함 모델", behavior_prob_test),
]:
    order = np.argsort(prob)

    for share in [0.1, 0.2]:
        selected = order[
            :int(np.ceil(len(y_test_compare) * share))
        ]

        non_return_count = (
            y_test_compare[selected] == 0
        ).sum()

        selected_non_return_rate = (
            non_return_count / len(selected)
        )

        overall_non_return_rate = (
            y_test_compare == 0
        ).mean()

        risk_rows.append({
            "모델": model_name,
            "위험군": f"상위 {share:.0%}",
            "선정 인원": len(selected),
            "실제 미재완료자": non_return_count,
            "실제 미재완료율": selected_non_return_rate,
            "미재완료자 포착률": (
                non_return_count
                / (y_test_compare == 0).sum()
            ),
            "Lift": (
                selected_non_return_rate
                / overall_non_return_rate
            ),
        })

test_risk_comparison = pd.DataFrame(risk_rows)

display(
    test_risk_comparison.round(4)
)

print(
    "테스트 전체 미재완료율:",
    f"{(y_test_compare == 0).mean():.2%}"
)

,모델,위험군,선정 인원,실제 미재완료자,실제 미재완료율,미재완료자 포착률,Lift
0,기준 모델,상위 10%,87,82,0.9425,0.1080,1.0804
1,기준 모델,상위 20%,174,156,0.8966,0.2055,1.0277
2,24시간 행동 포함 모델,상위 10%,87,85,0.9770,0.1120,1.1199
3,24시간 행동 포함 모델,상위 20%,174,165,0.9483,0.2174,1.0870


테스트 전체 미재완료율: 87.24%


### 12.1 테스트셋 비교 결과

첫 완료 후 24시간 행동을 포함한 모델은 기준 모델보다 테스트 데이터에서
모든 평가 지표가 개선되었다.

- PR-AUC: 0.1574 → 0.2201
- ROC-AUC: 0.5735 → 0.6489
- Brier Score: 0.1203 → 0.1133
- 위험 상위 10% 미재완료자: 82명 → 85명
- 위험 상위 20% 미재완료자: 156명 → 165명

특히 위험 상위 20%에서는 동일하게 174명을 선정하면서도,
24시간 행동 포함 모델이 실제 미재완료자를 9명 더 포착했다.

따라서 학교, 최초 완료 시점, 활성화에 걸린 시간과 최초 완료 당시 친구 수를
고려한 뒤에도 첫 완료 후 24시간 행동은 D14 재완료 예측에 추가 정보를 제공했다.

다만 이는 예측 관계이며, 해당 행동을 증가시키면 D14 재완료율이 상승한다는
인과관계를 의미하지 않는다.

## 13. 첫 24시간 행동 강도별 D14 재완료율

예측에 추가 정보를 제공한 첫 24시간 행동이 어느 수준에서
D14 재완료율 차이로 나타나는지 구간별로 확인한다.

보낸 투표 수는 추가 질문 세트 완료 횟수와 구조적으로 중복될 수 있으므로,
질문 반복 참여, 받은 사회적 반응, 관계 확장의 세 가지 행동으로 구분한다.

In [56]:
# 구간별 표 생성
behavior_analysis_df = model_df.copy()

# 1. 추가 질문 세트 완료 횟수
behavior_analysis_df["반복_완료_구간"] = pd.cut(
    behavior_analysis_df["repeat_set_count_24h"],
    bins=[-1, 0, 2, 5, np.inf],
    labels=["0회", "1~2회", "3~5회", "6회 이상"],
)

# 2. 받은 투표 수
behavior_analysis_df["받은_투표_구간"] = pd.cut(
    behavior_analysis_df["received_vote_count_24h"],
    bins=[-1, 0, 14, 29, 51, np.inf],
    labels=[
        "0건",
        "1~14건",
        "15~29건",
        "30~51건",
        "52건 이상",
    ],
)

# 3. 새로 추가된 동급생 친구 수
behavior_analysis_df["신규_친구_구간"] = pd.cut(
    behavior_analysis_df["new_class_friends_24h"],
    bins=[-1, 0, 2, 5, np.inf],
    labels=["0명", "1~2명", "3~5명", "6명 이상"],
)


def summarize_d14_by_group(df, group_col):
    result = (
        df.groupby(group_col, observed=True)
        .agg(
            사용자=("user_id", "size"),
            D14_재완료자=("y_d14", "sum"),
            D14_재완료율=("y_d14", "mean"),
        )
        .reset_index()
    )

    result["D14_재완료율"] = (
        result["D14_재완료율"] * 100
    ).round(2)

    return result


print("① 추가 질문 세트 완료 횟수")
display(
    summarize_d14_by_group(
        behavior_analysis_df,
        "반복_완료_구간",
    )
)

print("② 받은 투표 수")
display(
    summarize_d14_by_group(
        behavior_analysis_df,
        "받은_투표_구간",
    )
)

print("③ 새로 추가된 동급생 친구 수")
display(
    summarize_d14_by_group(
        behavior_analysis_df,
        "신규_친구_구간",
    )
)

① 추가 질문 세트 완료 횟수


,반복_완료_구간,사용자,D14_재완료자,D14_재완료율
0,0회,363,27,7.44
1,1~2회,752,103,13.70
2,3~5회,1174,185,15.76
3,6회 이상,2041,440,21.56


② 받은 투표 수


,받은_투표_구간,사용자,D14_재완료자,D14_재완료율
0,0건,80,7,8.75
1,1~14건,996,175,17.57
2,15~29건,1069,196,18.33
3,30~51건,1089,191,17.54
4,52건 이상,1096,186,16.97


③ 새로 추가된 동급생 친구 수


,신규_친구_구간,사용자,D14_재완료자,D14_재완료율
0,0명,823,122,14.82
1,1~2명,1311,240,18.31
2,3~5명,1318,238,18.06
3,6명 이상,878,155,17.65


구간은 현재 탐색적 비교를 위해 임시로 설정했으며, 모델 학습에는 구간화하지 않은 원래 횟수형 변수를 사용했다. 최종 구간은 전체 분포와 구간별 표본 수를 확인한 뒤 조정할 예정이다.

### 13.1 결과 해석

- **추가 질문 세트 완료가 가장 강한 신호였다.**  
  완료 횟수가 `0회 7.44% → 1~2회 13.70% → 3~5회 15.76% → 6회 이상 21.56%`로 증가했다.  
  특히 6회 이상 완료한 사용자의 D14 재완료율은 0회 사용자보다 **14.12%p 높고 약 2.9배**였다.

- **받은 투표는 ‘많이 받는 것’보다 ‘한 번이라도 받는 것’이 중요해 보였다.**  
  0회 사용자는 8.75%였지만, 1회 이상 받은 구간은 약 17~18%였다.  
  다만 투표 수가 많아질수록 재완료율이 계속 상승하지는 않았다. 또한 0회 사용자는 80명으로 표본이 작아 해석에 주의해야 한다.

- **새로운 같은 학급 친구도 비슷한 문턱 효과를 보였다.**  
  0명은 14.82%, 1~2명은 18.31%였지만 이후에는 18% 내외로 큰 차이가 없었다.  
  즉, 친구를 계속 많이 추가하는 것보다 **새로운 연결이 한 번이라도 생기는 것**이 주요 신호로 보인다.

### 핵심 결론

첫 질문 세트 완료 후 24시간 동안 **질문 세트를 반복해서 완료하는 행동**이 D14 재완료와 가장 뚜렷하게 연결됐다.  
소셜 행동은 양이 계속 증가하는 것보다 **투표를 한 번이라도 받고 새 친구가 한 명이라도 생기는 경험**이 더 중요해 보였다.  
따라서 초기 24시간 내 두 번째 질문 참여와 최소한의 소셜 반응을 유도하는 방향을 우선 검토할 수 있다.

단, 이는 인과관계가 아니라 행동과 D14 재완료 사이의 연관성이므로 실제 효과는 A/B 테스트로 확인해야 한다.